# 05 · Patrones puntuales: ¿cómo se distribuyen los nuevos frentes mineros?

Cada hexágono donde la fracción perturbada creció más de 0.3 entre el primer y el último año se
trata como un **evento**. Sobre ese patrón de puntos:

- Intensidad λ y estimación por **kernel (KDE)**.
- Función **G** (vecino más cercano) e índice de **Clark-Evans**.
- Función **K de Ripley** / **L(r) − r** con envolventes de Monte Carlo bajo CSR.
- Dependencia de una covariable: distancia al río.

In [ ]:
%pip install -q rasterio pyproj h3 geopandas shapely esda libpysal spreg pykrige scikit-gstat mapclassify
dbutils.library.restartPython()

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from geocloud import spatial

feat = spark.table(dbx.tbl("gold_h3_features")).toPandas()
YEARS = sorted(int(c.split("_")[1]) for c in feat.columns if c.startswith("ndvi_") and c.split("_")[1].isdigit())
Y0, Y1 = YEARS[0], YEARS[-1]
gdf = spatial.h3_to_geodataframe(feat, crs_out=C.CRS_UTM)
print(f"Hexágonos: {len(gdf):,} | años: {YEARS}")

In [ ]:
UMBRAL = 0.3
ev = gdf[gdf["d_pert"] >= UMBRAL]
pts = ev[["x_km", "y_km"]].values
b = np.array(C.GRID_BOUNDS_UTM) / 1000
area = (b[2] - b[0]) * (b[3] - b[1])
lam = len(pts) / area
print(f"Eventos: {len(pts):,} | ventana: {area:,.0f} km² | λ = {lam:.3f} eventos/km²")

In [ ]:
from scipy.stats import gaussian_kde
kde = gaussian_kde(pts.T, bw_method="scott")
gx, gy = np.meshgrid(np.linspace(b[0], b[2], 260), np.linspace(b[1], b[3], 160))
dens = kde(np.vstack([gx.ravel(), gy.ravel()])).reshape(gx.shape) * len(pts)

fig, ax = plt.subplots(figsize=(13, 7))
im = ax.imshow(dens, origin="lower", extent=[b[0], b[2], b[1], b[3]], cmap="inferno")
ax.scatter(pts[:, 0], pts[:, 1], s=0.5, c="cyan", alpha=0.4)
plt.colorbar(im, ax=ax, shrink=0.6, label="eventos / km²")
ax.set_title(f"Intensidad estimada por kernel de nuevos frentes de perturbación ({Y0}-{Y1})")
ax.set_xlabel("UTM 19S Este (km)"); ax.set_ylabel("UTM 19S Norte (km)")
plt.tight_layout(); plt.savefig(FIG / "05_kde.png", dpi=150); plt.show()

## Funciones G y K

Bajo aleatoriedad espacial completa (CSR): $G(r) = 1 - e^{-\lambda\pi r^2}$ y $K(r) = \pi r^2$,
por lo que $L(r) - r = \sqrt{K(r)/\pi} - r = 0$. Valores por encima de la envolvente indican
**agrupamiento** a esa escala.

*Nota:* los eventos son centroides de una grilla hexagonal (separación ≈ 0.35 km), así que las
escalas útiles empiezan alrededor de 0.5 km.

In [ ]:
from scipy.spatial import cKDTree
rng = np.random.default_rng(42)
P = pts[rng.choice(len(pts), size=min(3000, len(pts)), replace=False)]
nn = cKDTree(P).query(P, k=2)[0][:, 1]
lam_s = len(P) / area
R_ce = nn.mean() / (0.5 / np.sqrt(lam_s))
r = np.linspace(0, 3, 60)
G_emp = np.array([(nn <= ri).mean() for ri in r])
G_csr = 1 - np.exp(-lam_s * np.pi * r**2)

radii = np.linspace(0.5, 10, 30)
K = spatial.ripley_k(P, tuple(b), radii)
lo, hi = spatial.ripley_envelope(len(P), tuple(b), radii, n_sim=99)
L = lambda k: np.sqrt(k / np.pi) - radii

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(r, G_emp, label="G empírica"); ax[0].plot(r, G_csr, "--", label="CSR")
ax[0].set_xlabel("r (km)"); ax[0].set_title(f"Función G | Clark-Evans R = {R_ce:.3f}"); ax[0].legend()
ax[1].fill_between(radii, L(lo), L(hi), color="grey", alpha=0.3, label="envolvente CSR (99 sim.)")
ax[1].plot(radii, L(K), c="r", label="L(r) − r observada")
ax[1].axhline(0, c="k", lw=0.6); ax[1].set_xlabel("r (km)"); ax[1].legend()
ax[1].set_title("Función L de Ripley")
plt.tight_layout(); plt.savefig(FIG / "05_G_L.png", dpi=150); plt.show()
print(f"Clark-Evans R = {R_ce:.3f} (R < 1: agrupamiento)")

## Dependencia de la covariable: distancia al río

In [ ]:
from scipy.stats import ks_2samp
if "dist_rio_km" in gdf:
    a, bb = ev["dist_rio_km"].dropna(), gdf["dist_rio_km"].dropna()
    ks = ks_2samp(a, bb)
    fig, ax = plt.subplots(figsize=(8, 5))
    for serie, lab in [(a, "eventos"), (bb, "todo el territorio")]:
        xs = np.sort(serie); ax.plot(xs, np.arange(1, len(xs) + 1) / len(xs), label=lab)
    ax.set_xlabel("Distancia al río (km)"); ax.set_ylabel("Función de distribución acumulada")
    ax.set_title(f"Kolmogorov-Smirnov D = {ks.statistic:.3f}, p = {ks.pvalue:.1e}"); ax.legend()
    plt.tight_layout(); plt.savefig(FIG / "05_ecdf_rio.png", dpi=150); plt.show()
    print(f"Mediana de distancia al río: eventos {a.median():.2f} km vs territorio {bb.median():.2f} km")